# Week 6 Homework: Distillation as Recipe Design

**ECBS5200 — Practical Deep Learning Engineering**

Closing homework of the term. **Budget ~6–7 hours** including memo.
Roughly: Part 1 (teacher diagnosis) ~45 min, Part 2 (recipe shortlist
+ grid selection + eval) ~2 h, Part 3 (two literature tests) ~45 min,
Part 4 (memo, 5 sections with cross-week integration and reference
tables) ~2.5–3 h. Start the memo early; the synthesis takes time.

## The frame

Distillation is one of several ways to transfer model properties.
Different recipes transfer different things at different costs. The
applied ML skill is **naming the property you need, picking the
cheapest recipe that delivers it, and knowing when no cheap
substitute exists.**

Specifically: post-hoc temperature scaling reproduces top-1
calibration almost for free — but it can't reshape the per-class
probability distribution the way distillation can. So "should I use
KD?" is the wrong question. The right question is "what property does
my deployment depend on, and what's the cheapest recipe that delivers
it?"

## Five parts

- **Part 0 (~5 min):** pick a deployment scenario + accept the data budget
- **Part 1 (~45 min):** diagnose what the teacher has worth transferring
- **Part 2 (~2h):** rank properties for your scenario, build a recipe shortlist
- **Part 3 (~45 min):** test two mechanism claims from the literature
- **Part 4 (~1.5h):** choose a recipe, defend, write the memo

No GPU training required. Analysis-only on pre-computed artifacts.
Run on Kaggle CPU or your laptop — your call.

## Setup

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → CPU is fine (this homework is analysis-only)
4. **Secrets** → add `HF_TOKEN` (read token sufficient — no uploads)

In [1]:
# GUIDED: env config. Run as-is.
import os, sys, subprocess

if os.path.exists("/kaggle/working"):
    os.environ["HF_HOME"] = "/kaggle/working/.hf_cache"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# subprocess.check_call([
#     sys.executable, "-m", "pip", "install", "--quiet",
#     "transformers>=4.53", "datasets", "scikit-learn", "matplotlib", "pandas",
# ])
# subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
#                check=False)
# print("Packages installed.")

In [2]:
# GUIDED: HuggingFace auth. Run as-is.
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv(dotenv_path=".env", override=False)
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: no HF_TOKEN found — public reads still work but rate limits hit fast.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace.


In [3]:
# GUIDED: imports + helpers. Run as-is.
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

def softmax(logits):
    s = logits - logits.max(axis=-1, keepdims=True)
    e = np.exp(s); return e / e.sum(axis=-1, keepdims=True)

def expected_calibration_error(probs, labels, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    confs = probs.max(axis=-1); preds = probs.argmax(axis=-1)
    correct = (preds == labels); n = len(labels); ece = 0.0
    for i in range(n_bins):
        m = (confs > edges[i]) & (confs <= edges[i + 1])
        if m.sum() > 0:
            ece += (m.sum() / n) * abs(correct[m].mean() - confs[m].mean())
    return float(ece)

def nll_fn(probs, labels):
    return float(-np.log(probs[np.arange(len(labels)), labels].clip(min=1e-12)).mean())

def js_divergence(p, q, eps=1e-12):
    """Jensen-Shannon divergence between two probability distributions
    (per-row if 2D). Returns scalar (mean over rows if 2D).
    Symmetric, bounded in [0, ln(2)]."""
    p = np.clip(p, eps, 1.0); q = np.clip(q, eps, 1.0)
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * (np.log(p) - np.log(m)), axis=-1)
    kl_qm = np.sum(q * (np.log(q) - np.log(m)), axis=-1)
    return float(np.mean(0.5 * (kl_pm + kl_qm)))

print("Imports + helpers OK.")

Imports + helpers OK.


In [4]:
# GUIDED: load all the artifacts you'll need across the homework.
# Vanilla and distilled students from the lab; teacher logits from
# Week 6 dataset repo; six grid configs from the homework grid.

# Lab artifacts (vanilla + distilled student val predictions)
VANILLA_REPO   = "earino/ecbs5200-week6-vanilla-baseline"
DISTILLED_REPO = "earino/ecbs5200-week6-distilled-student"

v_path = hf_hub_download(repo_id=VANILLA_REPO, repo_type="model", filename="val_predictions.npz")
d_path = hf_hub_download(repo_id=DISTILLED_REPO, repo_type="model", filename="val_predictions.npz")
v_npz = np.load(v_path); d_npz = np.load(d_path)

v_logits = v_npz["logits"].astype(np.float32); v_preds = v_npz["preds"]
v_labels = v_npz["labels"]; val_tiers = v_npz["val_tiers"]
d_logits = d_npz["logits"].astype(np.float32); d_preds = d_npz["preds"]
assert (d_npz["labels"] == v_labels).all() and (d_npz["val_tiers"] == val_tiers).all()

# Teacher val logits (from the public dataset repo built for the lab)
TEACHER_REPO = "earino/ecbs5200-week6-teacher-logits"
t_path = hf_hub_download(repo_id=TEACHER_REPO, repo_type="dataset",
                         filename="val_logits_canonical_final.npz")
t_npz = np.load(t_path)
t_logits = t_npz["logits"].astype(np.float32); t_labels = t_npz["labels"]
assert (t_labels == v_labels).all(), "teacher and student val orderings differ"

# Tier label sets
HEAD_LABELS = sorted(set(v_labels[val_tiers == "head"].tolist()))
MID_LABELS  = sorted(set(v_labels[val_tiers == "mid"].tolist()))
TAIL_LABELS = sorted(set(v_labels[val_tiers == "tail"].tolist()))
NUM_LABELS = 113

# Grid (6 configs)
GRID_REPO = "earino/ecbs5200-week6-grid-results"
GRID_CONFIGS = [(1.0, 0.7), (4.0, 0.7), (8.0, 0.7), (1.0, 0.9), (4.0, 0.9), (8.0, 0.9)]
def grid_filename(T_d, alpha):
    return f"grid_Td{int(T_d)}_a{int(round(alpha * 100))}_val_predictions.npz"

grid_npz = {}
for T_d, alpha in GRID_CONFIGS:
    p = hf_hub_download(repo_id=GRID_REPO, repo_type="dataset",
                        filename=grid_filename(T_d, alpha))
    grid_npz[(T_d, alpha)] = np.load(p)
print(f"All artifacts loaded.  vanilla/distilled/teacher + {len(grid_npz)} grid configs.")
print(f"|head|={len(HEAD_LABELS)}  |mid|={len(MID_LABELS)}  |tail|={len(TAIL_LABELS)}")

All artifacts loaded.  vanilla/distilled/teacher + 6 grid configs.
|head|=20  |mid|=40  |tail|=53


---

# Part 0 — Pick your deployment scenario (~5 min)

You will defend ONE deployment recommendation. Different deployments
care about different properties. Pick one of the three scenarios
below and commit to it before doing the analysis. **Don't switch
scenarios mid-homework** — the rest of the analysis is anchored to
the property priorities your scenario fixes.

| Scenario | Hard constraint | Primary metric | Secondary metric |
|---|---|---|---|
| **A — High-throughput batch triage** | <10 ms/example at batch=32 on T4; serve at 10k QPS | Macro F1 | Per-example cost |
| **B — Regulated escalation review** | Human reviewer sees model's top-1 confidence to decide whether to escalate; calibrated probabilities required | ECE (target ≤0.05) | NLL |
| **C — Long-tail rare-class monitoring** | Tail-class detection matters; false-negatives on tail are expensive | Tail F1 | Tail calibration |

**Note on Scenario C — read before picking it.** Scenario C is
intentionally hard. The data ceiling we've measured all term may
mean **no recipe within the data budget cleanly wins** Scenario C.
That is a legitimate, fully-credited answer: a defense that says
"the binding constraint is data, not recipe; here are the numbers
that show why; here is the experiment that would change the answer"
scores as well as any "recipe X wins" defense for the other
scenarios. Don't avoid C because it's harder to "solve"; pick C if
the problem is interesting to you and defend the constraint.

**Data budget constraint.** Your scenario has a fixed labeled
dataset — the 79,278 train+test examples already used for both the
vanilla and distilled students. You may NOT recommend "get more
data." The lab found that data shift contributed ~+0.055 macro F1
while distillation contributed ~+0.015 — a 3.6× ratio. In industry,
getting more high-quality labels is often impossible (annotation
cost, regulatory constraints, rare events). Design within what you
have.

**YOUR CHOICE:** A / B / C: B

**YOUR REASONING (one sentence — why this scenario):** I picked B because the deployment uses top-1 confidence and calibration is the main risk.

**YOUR INFORMAL PRIOR (before doing any analysis): which recipe do
you think will win for your scenario?** (Don't be precise; just write
the gut feeling. Predict-then-observe across the whole homework — we
come back to this in Part 4.)

**MY INFORMAL PRIOR:** vanilla + temperature scaling will win on ECE.

---

# Part 1 — Diagnose the teacher (~45 min)

Goal: name **3 or more properties** the teacher has that are
potentially worth transferring to the student. **Unranked.** Ranking
happens in Part 2 once you've committed to a scenario's priorities.

For each property, you need:

1. A **name** (what is the property?)
2. A **measurement** (how do you know the teacher has it?)
3. A **transferability assessment** (is this plausibly transferable to a
   smaller student, or is it blocked by something — e.g., the data
   ceiling on tail classes)?

### 1a — Compute the teacher's diagnostic metrics

In [5]:
# INTERACTIVE: per-class F1 for both teacher and the two students.
# Length-113 numpy arrays.
t_preds = t_logits.argmax(axis=-1)
t_per_class_f1 = f1_score(t_labels, t_preds, average=None, labels=range(NUM_LABELS), zero_division=0)
v_per_class_f1 = f1_score(v_labels, v_preds, average=None, labels=range(NUM_LABELS), zero_division=0)
d_per_class_f1 = f1_score(v_labels, d_preds, average=None, labels=range(NUM_LABELS), zero_division=0)

# Distilled - vanilla per-class delta (you'll need this in 1b and Part 2)
per_class_delta = d_per_class_f1 - v_per_class_f1

# Teacher per-tier F1 + per-tier ECE
t_probs = softmax(t_logits)
print(f"Teacher per-tier diagnostics:")
for tier_name, tier_labels in [("head", HEAD_LABELS), ("mid", MID_LABELS), ("tail", TAIL_LABELS)]:
    m = val_tiers == tier_name
    f1 = f1_score(t_labels[m], t_preds[m], labels=tier_labels, average="macro", zero_division=0)
    ece = expected_calibration_error(t_probs[m], t_labels[m])
    nll = nll_fn(t_probs[m], t_labels[m])
    print(f"  {tier_name:5s}  F1 {f1:.4f}  ECE {ece:.4f}  NLL {nll:.4f}  (n={m.sum()})")

Teacher per-tier diagnostics:
  head   F1 0.6522  ECE 0.0250  NLL 0.9749  (n=5155)
  mid    F1 0.4498  ECE 0.1320  NLL 2.2038  (n=1065)
  tail   F1 0.1976  ECE 0.2261  NLL 3.3409  (n=210)


### 1b — Inspect the teacher's confidence shape

Some properties of the teacher only show up in the *full distribution*,
not in the argmax. For 5 random val examples drawn from each tier,
print the teacher's top-3 classes and their probabilities. Look at
the *relative magnitudes*, not just whether the top class is right.

In [6]:
# GUIDED: print teacher top-3 distributions for sample examples per tier.
rng = np.random.RandomState(SEED)
for tier_name in ["head", "mid", "tail"]:
    idxs = np.where(val_tiers == tier_name)[0]
    sample = rng.choice(idxs, size=min(5, len(idxs)), replace=False)
    print(f"\n{tier_name.upper()} examples — teacher top-3:")
    for i in sample:
        true_class = int(t_labels[i])
        top3 = np.argsort(-t_probs[i])[:3]
        s = ", ".join(f"c{int(c)}={t_probs[i, c]:.3f}" for c in top3)
        correct = "✓" if int(t_preds[i]) == true_class else "✗"
        print(f"  idx={i:5d}  true=c{true_class:3d}  {correct}  top3: {s}")


HEAD examples — teacher top-3:
  idx= 5288  true=c 52  ✓  top3: c52=0.992, c32=0.002, c51=0.002
  idx=  609  true=c 54  ✗  top3: c92=0.275, c54=0.194, c56=0.099
  idx= 5046  true=c 37  ✗  top3: c23=0.516, c111=0.270, c37=0.076
  idx= 3426  true=c 56  ✓  top3: c56=0.698, c102=0.284, c96=0.003
  idx= 3432  true=c 10  ✓  top3: c10=0.902, c52=0.028, c37=0.015

MID examples — teacher top-3:
  idx=  862  true=c 94  ✗  top3: c52=0.246, c79=0.202, c10=0.120
  idx= 2523  true=c 36  ✓  top3: c36=0.413, c59=0.145, c69=0.138
  idx= 1948  true=c 15  ✗  top3: c61=0.404, c15=0.116, c88=0.065
  idx= 4776  true=c 94  ✗  top3: c6=0.534, c8=0.279, c51=0.078
  idx=  627  true=c 36  ✗  top3: c60=0.230, c59=0.048, c36=0.039

TAIL examples — teacher top-3:
  idx= 3882  true=c 89  ✓  top3: c89=0.214, c72=0.184, c15=0.166
  idx= 3969  true=c 18  ✓  top3: c18=0.501, c67=0.092, c13=0.068
  idx= 4985  true=c 12  ✓  top3: c12=0.522, c22=0.139, c10=0.091
  idx=  414  true=c 73  ✗  top3: c0=0.383, c67=0.191, c53=0.

### 1c — Build the transferable properties spec

Fill in **3–5 properties** the teacher has. Use what you computed
above. Don't rank yet — Part 2 ranks against your scenario. Each row
should be one property.

**Example row (don't copy verbatim — write your own):**

- **Property:** "calibrated top-1 confidence"
- **Measurement:** teacher ECE 0.021 on val (post-temperature-scaling)
- **Transferable?** YES, plausibly via KD (transfers shape) OR via post-hoc temperature scaling on the student (~free)

**YOUR PROPERTIES (write at least 3):**

1. **Property:** Strong head-tier accuracy.
   - **Measurement:** head F1 0.6522 (n=5155).
   - **Transferable?** YES, KD should transfer this because head has lots of data.

2. **Property:** Top-1 calibration on head.
   - **Measurement:** head ECE 0.0250 (teacher).
   - **Transferable?** YES, via KD or cheap post-hoc temperature scaling.

3. **Property:** Full-distribution shape (soft targets) in mid/tail.
   - **Measurement:** mid NLL 2.2038 and tail NLL 3.3409; tail F1 0.1976.
   - **Transferable?** PARTIAL, KD can transfer shape but tail capacity is data-limited.

(Optional 4–5):

---

# Part 2 — Rank properties + build recipe shortlist (~2h)

Now that you've diagnosed the teacher and committed to a scenario,
rank the properties from Part 1 by deployment value FOR YOUR
SCENARIO, then design a candidate recipe shortlist.

### 2a — Rank properties for your scenario

Take the 3+ properties from Part 1. Order them by deployment value
for the scenario you picked in Part 0.

**YOUR RANKING (1 = most important for your scenario):**

1. **Property #1:** top-1 calibration (low ECE), **why this rank:** Scenario B makes escalation decisions from top-1 confidence, so calibration is the binding property.
2. **Property #2:** full-distribution shape (NLL / soft targets), **why this rank:** NLL is the secondary metric and matters if the downstream consumes probabilities.
3. **Property #3:** head-tier accuracy, **why this rank:** useful but not the main constraint for a calibrated escalation workflow.

Note: the same property list can rank very differently across
scenarios. Scenario A cares about throughput-friendly accuracy;
Scenario B cares about top-1 calibration; Scenario C cares about tail
behavior. The ranking IS the design choice.

### 2b — Build the recipe shortlist

Five recipes. Four are pre-computed; the fifth (wildcard) is a
config you propose but don't run. For each recipe, fill in the
table below.

**The five recipes:**

1. **Vanilla** — plain CE, no KD, no post-hoc fix. The Week 1
   baseline at full data scale.
2. **Vanilla + post-hoc temperature scaling** — fit a single T on
   a calibration fold, apply at inference. Cheap.
3. **Distilled (lab default, T_d=4, α=0.7)** — the lab's distilled
   student.
4. **One tuned grid config** — pick from the 6-config grid based on
   your scenario's primary metric. Justify the pick.
5. **Wildcard** — propose ONE recipe NOT in the grid (e.g.,
   `T_d=16`, `α=0.5`, distilled + temperature scaling, vanilla +
   label smoothing, anything from the readings). Predict from first
   principles where it would land. **Do not run it.** Justify the
   prediction.

**Example of a strong wildcard answer** (worked example for
calibration — pick a different one for your shortlist):

> *Wildcard: distilled + post-hoc temperature scaling.*
> Prediction: ECE drops below `vanilla + T` because KD fixes
> distributional shape (non-top-1 mass) while T fixes top-1
> sharpness — the two recipes correct different defects so they
> stack. Macro F1 stays roughly equal to distilled alone because
> T-scaling is argmax-invariant. **Failure mode:** if KD already
> over-flattens the distribution, an additional T > 1 over-flattens
> further and NLL gets worse. **What would flip the decision:** if
> Scenario A (throughput) — the extra inference step from T-scaling
> is negligible, ship the stacked recipe; if Scenario B (regulated)
> — the extra calibration check earns its keep; if Scenario C
> (tail) — no, the binding constraint is data, not calibration.

**Cost note for the table:** vanilla and distilled students share
the same architecture (ModernBERT-base, 149M). **Serving cost is
identical** — same forward pass, same latency, same memory. Cost
differences live in *training compute* (KD adds the precomputed
logit load + KL term computation during training, not inference).

**Recipe table (fill in for YOUR scenario's primary metric):**

| # | Recipe | Cost (train + serve) | Predicted property delivered | Measured value of primary metric | Ship for your scenario? |
|---|---|---|---|---|---|
| 1 | Vanilla | baseline train + same serve | baseline only | ECE 0.1308 | No (fails <=0.05) |
| 2 | Vanilla + temp | no extra train; tiny calib step; same serve | top-1 calibration | ECE 0.0263 | Yes |
| 3 | Distilled (4, 0.7) | extra train (KD); same serve | distribution shape + some calibration | ECE 0.0546 | No (over target) |
| 4 | Grid config: (T_d=1.0, α=0.9) | extra train; same serve | calibration-focused KD | ECE 0.0286 | No (temp is cheaper) |
| 5 | Wildcard: T_d=16, α=0.9 | n/a (didn't run) | softer targets with stronger KD weight | n/a (didn't run) | Maybe, but not without evidence |

### 2c — Verify recipes 1–4 with measurements

Compute the metrics for your scenario across the four runnable
recipes. Use the helpers; the cells below are GUIDED.

**Important — read before you read the table:** All four arms in
this section are evaluated on the **eval half of val (n=3,215)**, not
the full val set the lab used (n=6,430). This is necessary because
temperature scaling needs a held-out fit set — we fit T on the cal
half (n=3,215) and evaluate all four arms on the eval half so the
comparison is fair.

Consequence: your numbers here will differ from the lab's full-val
numbers by ~1–2 points on macro F1. That is expected and is not a
bug. The sanity check below confirms your loaded artifacts match
the lab's full-val numbers before we move to the eval-fold table.

In [7]:
# GUIDED: fit T on a 50/50 cal/eval split for vanilla.
n_val = len(v_labels)
rng_fold = np.random.RandomState(SEED + 999)
perm = rng_fold.permutation(n_val)
cal_idx = perm[: n_val // 2]
eval_idx = perm[n_val // 2 :]

def fit_temperature(logits, labels, max_iter=200):
    """LBFGS optimization of scalar T > 0 to minimize NLL.

    Parameterizes log_T so T = exp(log_T) is structurally positive — avoids
    the rare LBFGS pathology where T drifts non-positive on ill-behaved
    logits. Same pattern as Week 5's temperature_scale helper.
    """
    lt = torch.from_numpy(logits).float(); lab = torch.from_numpy(labels).long()
    log_T = torch.nn.Parameter(torch.zeros(1))  # log_T = 0 → T = 1.0 init
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=max_iter)
    nll_loss = torch.nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad()
        T = torch.exp(log_T)
        l = nll_loss(lt / T, lab); l.backward(); return l
    opt.step(closure)
    return float(torch.exp(log_T.detach()).item())

T_vanilla = fit_temperature(v_logits[cal_idx], v_labels[cal_idx])
print(f"Fitted T for vanilla student: {T_vanilla:.4f}")

# Sanity check: vanilla + distilled macro F1 on the FULL val set
# (should match the lab's 0.2638 / 0.2789 numbers to four decimals).
# If these don't match, your loaded artifacts are not the lab's.
_full_v_f1 = f1_score(v_labels, v_preds, average="macro", zero_division=0)
_full_d_f1 = f1_score(v_labels, d_preds, average="macro", zero_division=0)
print(f"Full-val sanity check — vanilla F1 {_full_v_f1:.4f} | "
      f"distilled F1 {_full_d_f1:.4f}")
print("(These should match the lab's full-val numbers; the table below "
      "uses the eval fold only, so its F1s will differ by 1–2 pts.)")

Fitted T for vanilla student: 1.3915
Full-val sanity check — vanilla F1 0.2638 | distilled F1 0.2789
(These should match the lab's full-val numbers; the table below uses the eval fold only, so its F1s will differ by 1–2 pts.)


In [8]:
# INTERACTIVE: pick your grid config. Replace the (T_d, alpha) below
# with your scenario-justified choice from the 6-config grid.
my_grid_config = (1.0, 0.9)  # picked for lowest ECE on eval fold
assert my_grid_config in GRID_CONFIGS, f"{my_grid_config} not in grid"
g_npz = grid_npz[my_grid_config]
g_logits = g_npz["logits"].astype(np.float32); g_preds = g_npz["preds"]

In [9]:
# GUIDED: compute the four runnable recipes' metrics on the eval fold.
# All four arms evaluated on the SAME eval fold for a fair comparison.
def eval_arm(logits_eval, labels_eval, T=1.0):
    probs = softmax(logits_eval / T)
    preds = probs.argmax(axis=-1)
    return {
        "macro_f1": float(f1_score(labels_eval, preds, average="macro", zero_division=0)),
        "ece": expected_calibration_error(probs, labels_eval),
        "nll": nll_fn(probs, labels_eval),
        "tail_f1": float(f1_score(
            labels_eval[val_tiers[eval_idx] == "tail"],
            preds[val_tiers[eval_idx] == "tail"],
            labels=TAIL_LABELS, average="macro", zero_division=0,
        )),
    }

vanilla_raw    = eval_arm(v_logits[eval_idx], v_labels[eval_idx])
vanilla_scaled = eval_arm(v_logits[eval_idx], v_labels[eval_idx], T=T_vanilla)
distilled_lab  = eval_arm(d_logits[eval_idx], v_labels[eval_idx])
grid_picked    = eval_arm(g_logits[eval_idx], v_labels[eval_idx])

print(f"{'Arm':<28}{'F1':>8}{'ECE':>10}{'NLL':>10}{'Tail F1':>10}")
print("-" * 66)
for name, m in [("Vanilla raw",          vanilla_raw),
                ("Vanilla + temp scaling", vanilla_scaled),
                ("Distilled (4, 0.7)",   distilled_lab),
                (f"Grid config {my_grid_config}", grid_picked)]:
    print(f"{name:<28}{m['macro_f1']:>8.4f}{m['ece']:>10.4f}"
          f"{m['nll']:>10.4f}{m['tail_f1']:>10.4f}")

Arm                               F1       ECE       NLL   Tail F1
------------------------------------------------------------------
Vanilla raw                   0.2833    0.1308    1.5515    0.1222
Vanilla + temp scaling        0.2833    0.0263    1.4297    0.1222
Distilled (4, 0.7)            0.2872    0.0546    1.3321    0.1217
Grid config (1.0, 0.9)        0.2896    0.0286    1.3370    0.1333


### 2d — Wildcard prediction (no measurement)

Propose your wildcard config. Justify the prediction from first
principles — what mechanism leads you to expect that result?

**YOUR WILDCARD:**

- **Recipe name:** Distilled with higher temperature and stronger KD weight (T_d=16, alpha=0.9).
- **Why this recipe is interesting for your scenario:** Scenario B wants low ECE, and the grid did not test targets softer than T_d=8. A higher T_d should make the teacher distribution less sharp, while alpha=0.9 keeps most of the loss on the soft-target KL term.
- **Predicted result vs the four measured recipes:** I would expect ECE around 0.03-0.04: better than distilled (0.0546), but probably not clearly better than vanilla+T (0.0263). NLL might stay close to the distilled/grid values around 1.33-1.36, and macro F1 would probably be flat or slightly down.
- **Mechanism for the prediction:** Higher T_d exposes more of the teacher's non-top-1 probability mass, and higher alpha makes the student follow that soft distribution more strongly. That could smooth top-1 confidence, but it could also over-soften the model, so I would not ship it without running the actual config.

### 2e — Reconcile your shortlist

Now look at your filled-in recipe table. **In writing**, answer:

1. **Which recipe ships for your scenario?**
2. **Why?** (Reference at least one specific number from your table.)
3. **Was your Part 0 informal prior right?** If not, what was your
   wrong assumption?

YOUR ANSWERS:

1. **Which recipe ships for your scenario?** Vanilla + temperature scaling.
2. **Why?** Vanilla+T has the best ECE in the measured table (0.0263). The picked grid config also meets the <=0.05 target at 0.0286, but it requires KD training and is slightly worse on ECE, while distilled alone misses the target at 0.0546. Macro F1 is unchanged by temperature scaling at 0.2833 because T-scaling does not change the argmax.
3. **Was your Part 0 informal prior right?** Yes, I expected vanilla + temp to win on ECE. I underestimated how close the calibration-focused grid config would get, but the cheap post-hoc recipe still wins for Scenario B.

---

# Part 3 — Test mechanism claims from the literature (~45 min)

Two specific claims, two specific tests. Each test is computable
from the artifacts you've already loaded.

### Test A — Stanton 2021: KD doesn't fully transfer the teacher's predictive distribution

**The claim** (Stanton et al. 2021, *Does Knowledge Distillation
Really Work?*, NeurIPS): even when KD improves the student's
generalization, the student does NOT fully recover the teacher's
predictive distribution. The mechanism is optimization difficulty,
not identifiability — the student *could* in principle match the
teacher, but optimization doesn't get there.

**The test:** measure the **Jensen-Shannon divergence** between the
teacher's softmax distribution and the student's softmax distribution,
averaged per tier. JS divergence is a symmetric, bounded distance
between probability distributions in `[0, ln 2] ≈ [0, 0.693]`.
Anchors: **JS = 0** means identical distributions; **JS ≈ 0.69** means
completely different (no overlap). Typical values for this task land
in the 0.05–0.25 range — read the *reduction* (`js_v − js_d`) as your
key number, not the absolute level. Compute for vanilla student vs
teacher, and distilled student vs teacher.

**The hypothesis:** distilled-vs-teacher JS should be SMALLER than
vanilla-vs-teacher JS (KD reduces the gap), but a substantial residual
remains, especially in the tail. KD reduces divergence; it doesn't
eliminate it.

Why this isn't argmax agreement: on a 113-class long-tail dataset,
both students will agree on the teacher's argmax for many head
examples by default. JS measures the *shape* of the distribution —
what Stanton actually argued the student fails to recover.

In [10]:
# GUIDED: compute teacher / vanilla / distilled softmax probabilities.
v_probs = softmax(v_logits)
d_probs = softmax(d_logits)
# t_probs already computed in Part 1

In [11]:
# INTERACTIVE: per-tier JS divergence between teacher and each student.
# Use the js_divergence helper (defined in setup).
print(f"{'Tier':<8}{'JS(teacher || vanilla)':>26}{'JS(teacher || distilled)':>28}{'reduction':>14}")
print("-" * 76)
for tier_name in ["head", "mid", "tail"]:
    m = val_tiers == tier_name
    js_v = js_divergence(t_probs[m], v_probs[m])
    js_d = js_divergence(t_probs[m], d_probs[m])
    reduction = js_v - js_d
    print(f"{tier_name:<8}{js_v:>26.4f}{js_d:>28.4f}{reduction:>+14.4f}")

Tier        JS(teacher || vanilla)    JS(teacher || distilled)     reduction
----------------------------------------------------------------------------
head                        0.0943                      0.0530       +0.0413
mid                         0.1522                      0.0901       +0.0620
tail                        0.1998                      0.1102       +0.0896


### Test A — Reconcile

Read your JS table:

1. **Did distillation reduce the JS divergence vs vanilla?** Per tier?
2. **Is there a residual?** What's the magnitude?
3. **Does the reduction depend on tier?** Is the dichotomy visible
   here (head vs tail)?
4. **Busbridge connection (no measurement needed):** the slides cited
   Busbridge et al. 2025 §E.8 — full-distribution KD vs top-1-only KD
   produces a 50-100× ECE difference at LLM scale. We did
   full-distribution KD here. If a hypothetical "hard-label student"
   had been trained on the teacher's argmax outputs only (no soft
   targets), what would you predict for its JS divergence to the
   teacher, *relative to* both our students? Reason from the
   mechanism — no need to run anything.

YOUR ANSWERS:

1. Yes, distillation reduced JS divergence in every tier. Head dropped from 0.0943 to 0.0530, mid from 0.1522 to 0.0901, and tail from 0.1998 to 0.1102.
2. There is still a clear residual after KD: 0.0530 on head, 0.0901 on mid, and 0.1102 on tail. So the distilled student moved closer to the teacher, but it is not a copy of the teacher distribution.
3. The reduction is largest on the tail in absolute terms (+0.0896), then mid (+0.0620), then head (+0.0413). The head-vs-tail pattern is visible because tail starts farther from the teacher and also keeps the biggest residual.
4. A hard-label student trained only on the teacher argmax would probably have higher JS divergence than the full-distribution distilled student, and maybe closer to vanilla. It would learn the teacher's top class but lose the teacher's probability shape across the other 112 classes, which is exactly what JS is measuring.

### Test B — Han Guo 2021: KD acts as a calibration regularizer in NLP

**The claim** (Han Guo et al. 2021, *Uncertainty Calibration for
Text Classification and the Role of Distillation*, RepL4NLP): KD's
calibration improvement reproduces a substantial fraction of what
you can get with post-hoc temperature scaling. They argue ~111% of
T-scaling's calibration win is reproducible by distillation alone.

**The test:** quantify the fraction of the distilled student's ECE
improvement that is reproducible by post-hoc temperature scaling on
the vanilla student. Three arms on the same eval fold:

- Arm 1: **Vanilla raw** (the baseline)
- Arm 2: **Vanilla + post-hoc T** (the cheap fix)
- Arm 3: **Distilled** (the expensive recipe)

Compute the three ECEs and the fraction:

`gap_v_to_vT = ECE_vanilla − ECE_vanilla_T` (post-hoc fix's gain)
`gap_v_to_d  = ECE_vanilla − ECE_distilled` (KD's gain)
`fraction_reproducible = gap_v_to_vT / gap_v_to_d`

Han Guo argues this fraction is large (close to 1, sometimes >1).
What do you find on this dataset?

In [12]:
# INTERACTIVE: compute the three-arm ECE comparison (you have the
# arms already — vanilla_raw, vanilla_scaled, distilled_lab from
# Part 2). Just lift the ECE values and compute the fraction.

ece_v   = vanilla_raw["ece"]
ece_vT  = vanilla_scaled["ece"]
ece_d   = distilled_lab["ece"]

gap_v_to_vT = ece_v - ece_vT
gap_v_to_d  = ece_v - ece_d
fraction = gap_v_to_vT / max(gap_v_to_d, 1e-9)

print(f"{'Arm':<28}{'ECE':>10}{'NLL':>10}{'Macro F1':>12}")
print("-" * 60)
print(f"{'Vanilla raw':<28}{vanilla_raw['ece']:>10.4f}{vanilla_raw['nll']:>10.4f}{vanilla_raw['macro_f1']:>12.4f}")
print(f"{'Vanilla + temp scaling':<28}{vanilla_scaled['ece']:>10.4f}{vanilla_scaled['nll']:>10.4f}{vanilla_scaled['macro_f1']:>12.4f}")
print(f"{'Distilled (4, 0.7)':<28}{distilled_lab['ece']:>10.4f}{distilled_lab['nll']:>10.4f}{distilled_lab['macro_f1']:>12.4f}")
print()
print(f"Δ ECE (vanilla → vanilla+T):    {gap_v_to_vT:+.4f}  (post-hoc fix's gain)")
print(f"Δ ECE (vanilla → distilled):    {gap_v_to_d:+.4f}  (KD's gain)")
print(f"Fraction reproducible by T:     {fraction*100:.0f}%")

Arm                                ECE       NLL    Macro F1
------------------------------------------------------------
Vanilla raw                     0.1308    1.5515      0.2833
Vanilla + temp scaling          0.0263    1.4297      0.2833
Distilled (4, 0.7)              0.0546    1.3321      0.2872

Δ ECE (vanilla → vanilla+T):    +0.1044  (post-hoc fix's gain)
Δ ECE (vanilla → distilled):    +0.0762  (KD's gain)
Fraction reproducible by T:     137%


### Test B — Reconcile (read carefully — this is metric-specific)

Look at the three-arm table. Then answer:

1. **What fraction of KD's ECE gain is reproducible by post-hoc T?**
   (Han Guo argues "most of it" — does that hold here?)
2. **Now look at NLL.** Does temperature scaling close the same
   fraction of the NLL gap? Or does the picture flip?
3. **Why might ECE and NLL behave differently?** Hint: ECE measures
   only top-1 confidence; NLL is sensitive to the entire 113-dim
   probability distribution. Temperature scaling is a uniform
   rescaling — what does that mean for what it can and can't fix?

YOUR ANSWERS:

1. Post-hoc T reproduces 137% of KD's ECE gain here. Vanilla raw ECE is 0.1308, vanilla+T is 0.0263, and distilled is 0.0546, so temperature scaling actually overshoots KD on top-1 calibration.
2. NLL flips the story. Vanilla raw NLL is 1.5515, vanilla+T is 1.4297, and distilled is 1.3321. Temperature scaling closes about 56% of the NLL gain from distillation: (1.5515 - 1.4297) / (1.5515 - 1.3321) ≈ 0.56.
3. ECE and NLL behave differently because ECE only looks at top-1 confidence calibration. Temperature scaling is a one-parameter sharpness fix, so it is well matched to ECE. NLL uses the full 113-class probability distribution, including how much mass goes to non-top-1 classes. A single temperature cannot change the relative ordering or shape among those classes, while KD can transfer more of that teacher distribution shape.

---

# Part 4 — Memo (~1.5h)

Five prompts. The rubric is in `assessments/week6_memo_rubric.md`.
Aim for 2–3 pages, HTML upload to Moodle by Wednesday morning before
next class. Tables and figures don't count toward the page limit.

**Reference-grade evidence is in the cells you already ran.** Don't
re-do anything; cite the numbers you computed.

## §3c carry-overs from the lab

Use the coverage_table you computed in the lab to fill these in. If
you didn't save it, the cell below recomputes.

In [13]:
# GUIDED: re-build the coverage table. `n_covered` is reported alongside
# coverage and accuracy because at high thresholds the covered count can
# drop into the dozens — accuracy 1.0 on 8 examples is descriptive, not
# a population estimate. Always cite n alongside.
def coverage_table(probs, labels, thresholds=(0.5, 0.6, 0.7, 0.8, 0.9)):
    confs = probs.max(axis=-1); preds = probs.argmax(axis=-1); rows = []
    for t in thresholds:
        cov_mask = confs >= t
        n_covered = int(cov_mask.sum())
        coverage = float(cov_mask.mean())
        accuracy = float((preds[cov_mask] == labels[cov_mask]).mean()) if n_covered > 0 else float("nan")
        rows.append((t, coverage, accuracy, n_covered))
    return rows

v_cov = coverage_table(softmax(v_logits), v_labels)
d_cov = coverage_table(softmax(d_logits), v_labels)
print(f"{'Thresh':<8}{'Van.cov':>9}{'Van.acc':>9}{'Van.n':>8}    "
      f"{'Dist.cov':>10}{'Dist.acc':>10}{'Dist.n':>8}")
print("-" * 72)
for (t, cv, av, nv), (_, cd, ad, nd) in zip(v_cov, d_cov):
    print(f"{t:<8.2f}{cv:>9.3f}{av:>9.3f}{nv:>8d}    "
          f"{cd:>10.3f}{ad:>10.3f}{nd:>8d}")

Thresh    Van.cov  Van.acc   Van.n      Dist.cov  Dist.acc  Dist.n
------------------------------------------------------------------------
0.50        0.807    0.696    5192         0.722     0.751    4641
0.60        0.708    0.739    4551         0.621     0.795    3994
0.70        0.606    0.781    3898         0.516     0.832    3315
0.80        0.500    0.829    3216         0.391     0.885    2516
0.90        0.365    0.878    2348         0.237     0.934    1522


### Q1. Vanilla coverage @ 0.70 vs distilled coverage @ 0.70
- Vanilla coverage @ 0.70: 0.606, accuracy: 0.781, n_covered: 3898
- Distilled coverage @ 0.70: 0.516, accuracy: 0.832, n_covered: 3315

### Q2. SLA: auto-routed examples must be ≥ 95% accurate
Report the **lowest** threshold from {0.5, 0.6, 0.7, 0.8, 0.9} where
accuracy ≥ 0.95. If no threshold in the table reaches 95% accuracy,
answer "none" and report the highest accuracy you observed.

- Vanilla: threshold none (best acc 0.878 at 0.90), coverage at that threshold 0.365, n_covered 2348
- Distilled: threshold none (best acc 0.934 at 0.90), coverage at that threshold 0.237, n_covered 1522

### Q3. Operational summary (one sentence)
What is the operational difference between a well-calibrated and a
poorly-calibrated model when the downstream system uses confidence
thresholds?

YOUR ANSWER: A well-calibrated model gives higher accuracy as you raise the confidence threshold (so coverage drops but accuracy rises), while a poorly calibrated model stays overconfident so accuracy does not track the threshold.

## Memo prompts (~half a page each, with tables/figures liberally)

Rubric weights: 25 / 15 / 15 / 15 / 30. Prompt 5 is the capstone
synthesis of the term — integrate every measurement axis from Weeks
1–6 into one defended deployment recommendation.

### Prompt 1 — What did distillation transfer? (25 pts)

Per-tier F1 deltas (distilled − vanilla) and per-tier ECE deltas from
the lab. Then: integrate Part 1 (your transferable properties spec),
Part 3 Test A (JS divergence per tier), and Part 3 Test B (the
metric-specific calibration finding).

**The framing the rubric rewards:** distillation transfers
*distributional structure* — the relative probabilities across all
113 classes. That property is **measurable in NLL but not in ECE**,
and **measurable in JS divergence but not in argmax agreement**. Show
this with your numbers.

**Response:** Distillation mainly transferred distributional structure, not just top-1 accuracy.
Per-tier F1 deltas (distilled - vanilla) were head +0.0209 (CI [+0.0065, +0.0356]), mid +0.0194 (CI [-0.0089, +0.0473]), tail +0.0049 (CI [-0.0262, +0.0401]).
ECE dropped in every tier: head 0.0849 -> 0.0127 (-0.0722), mid 0.2970 -> 0.1926 (-0.1044), tail 0.3932 -> 0.2574 (-0.1358).
JS divergence to teacher also dropped but did not vanish: head 0.0943 -> 0.0530, mid 0.1522 -> 0.0901, tail 0.1998 -> 0.1102.
That pattern fits the claim that KD transfers the *shape* of the distribution (relative probabilities), not just the argmax label.
Test B shows the metric split: vanilla+T beats KD on ECE (0.0263 vs 0.0546), but KD wins on NLL (1.3321 vs 1.4297).
So the property KD transfers that T-scaling cannot is the full-distribution structure measured by NLL and JS.
The teacher tail F1 is only 0.1976, so even perfect transfer cannot push tail much higher than that ceiling.


### Prompt 2 — What didn't distillation fix? (15 pts)

The data ceiling — tail F1 stays low for student, vanilla, AND
teacher. Defend: is this a property of the task, the data, or the
method? What additional measurements would you make to tell?

Required evidence:
- Cross-model tail F1: vanilla / distilled / teacher
- The data-confound finding (~+0.055 from data shift vs +0.015 from KD)
- At least one specific additional measurement that would falsify the
  data-ceiling hypothesis

**Response:** Tail F1 is low across models: vanilla 0.1249, distilled 0.1298, teacher 0.1976.
That cross-model pattern points to a data ceiling, not just a KD failure, because even the 32B teacher is only ~0.20 on tail.
The data-confound result backs this up: data shift gave about +0.055 macro F1 while KD gave only +0.015, so data composition moves the needle more than recipe choice.
If it were mainly a KD-method issue, we would expect a much larger jump from vanilla to distilled on tail, which we do not see.
To falsify the data-ceiling story, I'd run a tail-boost experiment: add ~50 labeled examples per tail class (or do a stratified resample that equalizes tail counts) and see if teacher tail F1 rises above ~0.30.
If it does, the ceiling was data; if it does not, the task or label space is likely too noisy or ambiguous for those rare classes.


### Prompt 3 — Hyperparameters and the noise floor (15 pts)

Use Part 2's recipe shortlist:
- Did the lab default `(T_d=4, α=0.7)` dominate any single metric?
- For your scenario's primary metric, was the spread across recipes
  inside or outside the bootstrap CI you measured in the lab?
- Anchor your answer to a specific tier and a specific bootstrap CI.
- What did your wildcard prediction add that the grid couldn't show?

**Response:** The lab default (T_d=4, alpha=0.7) does not dominate every metric I care about.
In the measured shortlist, the default has F1 0.2872, ECE 0.0546, NLL 1.3321, and tail F1 0.1217. The selected grid config (T_d=1.0, alpha=0.9) improves the Scenario B metric, with ECE 0.0286, and also has slightly higher F1 0.2896 and tail F1 0.1333, but its NLL is a bit worse at 1.3370.
For the noise floor, I anchor to the head-tier paired bootstrap CI from the lab: median +0.0210 with 95% CI [+0.0065, +0.0356], so the width is about 0.029.
The F1 difference between the selected grid config and the default is tiny (0.2896 - 0.2872 = 0.0024), far inside that width, so I would not claim a real best-F1 config from this one split.
By contrast, the ECE difference is much larger: 0.0546 for the default versus 0.0286 for the selected grid config, a drop of about 0.026. Since Scenario B is calibration-first, that is the separation I would pay attention to.
This supports the idea that F1 is mostly data-bounded here while calibration is more hyperparameter-sensitive.
The wildcard (T_d=16, alpha=0.9) adds a direction the grid did not test: much softer teacher targets with strong KD weight. It might lower ECE further, but it could also over-soften the student, so I would treat it as a follow-up experiment rather than a deployment choice.


### Prompt 4 — Three compressions, three lessons (15 pts)

Label compression (Week 4–5 pipeline reveal), weight compression
(Week 5 quantization), knowledge compression (Week 6).

- Which two interact most strongly on this dataset?
- Which two are nearly orthogonal?
- Defend each claim with a number.

**Response:** The strongest interaction is between weight compression and knowledge compression, because both change the deployed model's probability shape.
Quantization shifts accuracy and likely calibration: the decoder drops from macro F1 0.2401 (int8) to 0.2205 (int4) while latency improves (13.09 ms/ex -> 7.89 ms/ex).
KD, in contrast, raises macro F1 from 0.2638 to 0.2789 and drops ECE from 0.1300 to 0.0445 on full val.
If you quantize a distilled student, the int4 degradation could eat the KD gains or distort calibration, so those two techniques interact and need joint measurement.

The most orthogonal pair is label compression and weight compression.
Label compression is upstream data design: the merge map collapses 153 raw issues to 113 classes and MIN_CLASS_COUNT=5 drops the smallest labels.
Weight compression happens after training and mostly changes latency/VRAM (e.g., int8 encoder 6.27 ms/ex and 0.54 GB).
They operate at different stages, so they are nearly orthogonal in mechanism even though both matter in deployment.


### Prompt 5 — Defend the deployment (30 pts)

Pick one recipe — your shortlist winner — and defend it for the
scenario you committed to in Part 0.

Required evidence:

- The scenario you chose (A / B / C) and its primary + secondary
  metrics.
- **Specific numbers** from §3c (vanilla cov @ 0.70, distilled acc @
  0.70, the 95% SLA threshold for each model).
- Per-tier F1 / ECE from the lab.
- Your Part 3 finding stated **metric-by-metric**: how does
  temperature scaling compare to distillation on (a) ECE,
  (b) NLL, (c) macro F1, (d) JS divergence to teacher? They likely
  point in different directions — the right tool depends on which
  metric your scenario consumes.
- **Match metric to scenario.** If your scenario uses top-1
  confidence thresholds, ECE is the relevant metric. If your
  downstream consumes the full probability distribution, NLL is the
  relevant metric.
- One specific constraint that would flip your answer (named with a
  threshold and an alternative recipe).
- Cross-week reach: if your deployment also involves quantization,
  what does Week 5's findings imply about the combined recipe?

**This prompt is the capstone synthesis of the term.** A complete
answer here is the engineering document you'd write if a hiring
manager asked "what would you actually ship?"

**Response:** Scenario B (regulated escalation review). Primary metric is ECE (target <=0.05) and secondary is NLL. I would ship **vanilla + temperature scaling**.
On the eval fold, ECE is 0.0263 for vanilla+T versus 0.0546 for distilled and 0.1308 for vanilla. The selected grid config also meets the target at 0.0286, but vanilla+T is cheaper because it needs only a small calibration fit, not KD training.
Macro F1 is identical for vanilla and vanilla+T (0.2833), so we do not give up accuracy for the calibration gain. Distilled has slightly higher macro F1 (0.2872), and the grid config is 0.2896, but F1 is not the binding metric for Scenario B.
NLL is better for distilled (1.3321 vs 1.4297 for vanilla+T), but NLL is secondary here because the decision rule is based on top-1 confidence.

From §3c, vanilla coverage at 0.70 is 0.606 with accuracy 0.781 (n=3898), while distilled coverage at 0.70 is 0.516 with accuracy 0.832 (n=3315).
The 95% SLA is not met by either model: best vanilla accuracy is 0.878 at 0.90, and best distilled accuracy is 0.934 at 0.90. So we still escalate many cases, and calibration quality on the auto-routed subset is critical.
Per-tier F1 from the lab shows the same story: head 0.6105 -> 0.6314, mid 0.3797 -> 0.3990, tail 0.1249 -> 0.1298 (KD helps but not huge). Per-tier ECE improves with KD (head 0.0849 -> 0.0127, mid 0.2970 -> 0.1926, tail 0.3932 -> 0.2574), but temperature scaling still wins top-1 ECE on the eval fold.
Part 3 confirms the metric split: ECE favors vanilla+T, NLL favors KD, macro F1 changes only slightly, and JS divergence to the teacher favors KD (head 0.0943 -> 0.0530, mid 0.1522 -> 0.0901, tail 0.1998 -> 0.1102). Because Scenario B consumes top-1 confidence, I follow ECE.

**Constraint that flips my answer:** if the deployment starts using the full probability distribution (for example, a downstream Bayesian step) or we set a hard NLL target like <=1.35, I would switch to distilled (NLL 1.3321) or to a distilled + temperature-scaling follow-up if it kept the NLL advantage while lowering ECE. If the ECE requirement relaxed above ~0.06, distilled alone could also be acceptable.

**Cross-week reach:** if we also quantize, Week 5 shows int4 drops macro F1 (decoder 0.2401 at int8 to 0.2205 at int4) and likely adds calibration risk. For Scenario B, I would avoid int4 and at most use int8 after re-checking ECE on the exact final recipe.


---

### End of Week 6 homework

**Submit:** HTML export of this notebook with all interpretation
fields filled in, plus the memo as a separate HTML/PDF (linked from
the notebook is fine).

**Course closure:** the Week 6 memo is the final written deliverable
before the final exam. Prompt 5 is the capstone synthesis.